In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("GridSearchEmotionML").getOrCreate()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1748575578413_0002,pyspark,idle,Link,Link,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
%%configure -f
{
    "conf": {
        "spark.pyspark.python": "python3",
        "spark.pyspark.virtualenv.enabled": "true",
        "spark.pyspark.virtualenv.type":"native",
        "spark.pyspark.virtualenv.bin.path":"/usr/bin/virtualenv"
    }
}

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
1,application_1748575578413_0003,pyspark,idle,Link,Link,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
1,application_1748575578413_0003,pyspark,idle,Link,Link,✔


In [4]:
from pyspark.sql.functions import regexp_replace

df_base = spark.read.csv(
    "s3://vera-final-project-bucket/athena-query-results/final_joined_table.csv",
    header=True, inferSchema=True
)


df_base = df_base.withColumn("emotion_clean", regexp_replace("emotion", r"[\[\]']", ""))

df_base.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- track_id: string (nullable = true)
 |-- spotify_id: string (nullable = true)
 |-- emotion: string (nullable = true)
 |-- title: string (nullable = true)
 |-- artist: string (nullable = true)
 |-- id: string (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- analysis_url: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- duration: integer (nullable = true)
 |-- energy: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- liveness: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: integer (nullable = true)
 |-- valence: double (nullable = true)
 |-- emotion_clean: string (nullable = true)

In [5]:
sc.install_pypi_package("pandas==1.0.5", "https://pypi.org/simple")
sc.install_pypi_package("boto3", "https://pypi.org/simple")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.sql.functions import col, when
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

feature_cols = ['acousticness', 'danceability', 'energy', 'instrumentalness',
                'liveness', 'loudness', 'speechiness', 'valence', 'tempo']

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_unscaled")
scaler = StandardScaler(inputCol="features_unscaled", outputCol="features", withStd=True, withMean=True)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Parameter Settings
Random Forest Parameter Grid
    paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [100, 200]) \
    .addGrid(rf.maxDepth, [5, 10, 20]) \
    .addGrid(rf.minInstancesPerNode, [1, 2]) \
    .build()

MLP Parameter Grid
    paramGrid = ParamGridBuilder() \
    .addGrid(mlp.layers, [
    [input_dim, 100, 2],
    [input_dim, 50, 50, 2],
    [input_dim, 128, 64, 2]
    ]) \
    .addGrid(mlp.maxIter, [100, 300]) \
    .addGrid(mlp.stepSize, [0.01, 0.05]) \
    .build()

In [7]:
# Define EMR resource config — replace dynamically before each run if needed
INSTANCE_TYPE = "m5.xlarge"
INSTANCE_COUNT = 7

def get_static_emr_resources():
    return f"EMR, {INSTANCE_TYPE} x {INSTANCE_COUNT}"

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

#### Random Forest Parameter Grid

In [8]:
# RF - Grid Search

from pyspark.ml.classification import RandomForestClassifier

def run_rf_for_emotion(emotion_label):
    df = df_base.withColumn("label", when(col("emotion_clean").contains(emotion_label), 1).otherwise(0))
    
    df = assembler.transform(df)
    df = scaler.fit(df).transform(df)

    rf = RandomForestClassifier(labelCol="label", featuresCol="features", seed=42)
    
    paramGrid = ParamGridBuilder() \
        .addGrid(rf.numTrees, [100, 200]) \
        .addGrid(rf.maxDepth, [5, 10, 20]) \
        .addGrid(rf.minInstancesPerNode, [1, 2]) \
        .build()
    
    evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
    
    cv = CrossValidator(
        estimator=Pipeline(stages=[rf]),
        estimatorParamMaps=paramGrid,
        evaluator=evaluator,
        numFolds=3,
        parallelism=2
    )
    
    import time
    start = time.time()
    cv_model = cv.fit(df)
    duration = time.time() - start
    
    best_model = cv_model.bestModel.stages[-1]
    best_score = evaluator.evaluate(cv_model.transform(df))

    print(f"{emotion_label.upper()} finished: AUC = {best_score:.4f}, time = {duration:.2f}s")
    
    return {
        "Platform": "Spark",
        "Method": "Grid Search",
        "Model": "Random Forest",
        "Label": emotion_label,
        "Runtime (s)": round(duration, 2),
        "Best Score": round(best_score, 4),
        "Resources Used": get_static_emr_resources(),
        "Notes": f"numTrees={best_model.getOrDefault('numTrees')}, maxDepth={best_model.getOrDefault('maxDepth')}, minInstancesPerNode={best_model.getOrDefault('minInstancesPerNode')}"
    }


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [9]:
emotion_list = ["joy", "sadness", "anger", "fear", "surprise", "disgust"]
results = []

for emo in emotion_list:
    print(f"Running Random Forest tuning for: {emo}")
    results.append(run_rf_for_emotion(emo))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Running Random Forest tuning for: joy
JOY finished: AUC = 0.7601, time = 174.93s
Running Random Forest tuning for: sadness
SADNESS finished: AUC = 0.7434, time = 170.65s
Running Random Forest tuning for: anger
ANGER finished: AUC = 0.7397, time = 165.63s
Running Random Forest tuning for: fear
FEAR finished: AUC = 0.7373, time = 160.95s
Running Random Forest tuning for: surprise
SURPRISE finished: AUC = 0.8072, time = 164.11s
Running Random Forest tuning for: disgust
DISGUST finished: AUC = 0.7464, time = 160.48s

In [10]:
import pandas as pd
import boto3

df_result = pd.DataFrame(results)
local_path = "/tmp/spark_rf_gridsearch_results.csv"
df_result.to_csv(local_path, index=False)

s3 = boto3.client('s3')
bucket_name = 'vera-final-project-bucket'
output_key = 'gridsearch-results/spark_rf_gridsearch_results.csv'

with open(local_path, 'rb') as f:
    s3.put_object(Bucket=bucket_name, Key=output_key, Body=f)

print(f"Upload successful: s3://{bucket_name}/{output_key}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Upload successful: s3://vera-final-project-bucket/gridsearch-results/spark_rf_gridsearch_results.csv
/tmp/1748576344569-0/lib/python3.7/site-packages/boto3/compat.py:82: PythonDeprecationWarning: Boto3 will no longer support Python 3.7 starting December 13, 2023. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.8 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)

In [11]:
# RF - Random Search
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import itertools
import random
import pandas as pd
import boto3
import time

# Define the function for a single emotion
def run_rf_randomsearch_for_emotion(emotion_label):
    # Binary classification: whether this emotion is present
    df = df_base.withColumn("label", when(col("emotion_clean").contains(emotion_label), 1).otherwise(0))
    
    # Feature transformation
    df = assembler.transform(df)
    df = scaler.fit(df).transform(df)

    # Initialize model
    rf = RandomForestClassifier(labelCol="label", featuresCol="features", seed=42)

    # Define parameter search space
    param_space = {
        rf.numTrees: [50, 100, 150, 200],
        rf.maxDepth: [5, 10, 15, 20],
        rf.minInstancesPerNode: [1, 2, 3]
    }

    # Generate all combinations and randomly sample 10
    all_combinations = list(itertools.product(*param_space.values()))
    sampled_combinations = random.sample(all_combinations, k=10)

    # Convert to Spark param maps
    param_maps = []
    for combo in sampled_combinations:
        param_maps.append({
            list(param_space.keys())[0]: combo[0],
            list(param_space.keys())[1]: combo[1],
            list(param_space.keys())[2]: combo[2]
        })

    # Evaluator
    evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

    # CrossValidator with random sampled param grid
    cv = CrossValidator(
        estimator=Pipeline(stages=[rf]),
        estimatorParamMaps=param_maps,
        evaluator=evaluator,
        numFolds=3,
        parallelism=2
    )

    # Train and time
    start = time.time()
    cv_model = cv.fit(df)
    duration = time.time() - start

    # Extract best model and AUC
    best_model = cv_model.bestModel.stages[-1]
    best_score = evaluator.evaluate(cv_model.transform(df))

    # Extract best params safely
    num_trees = best_model.getNumTrees
    max_depth = best_model.getOrDefault(best_model.maxDepth)
    min_instances = best_model.getOrDefault(best_model.minInstancesPerNode)

    print(f"{emotion_label.upper()} Random Search: AUC = {best_score:.4f}, Time = {duration:.2f}s")

    return {
        "Platform": "Spark",
        "Method": "Random Search",
        "Model": "Random Forest",
        "Label": emotion_label,
        "Runtime (s)": round(duration, 2),
        "Best Score": round(best_score, 4),
        "Resources Used": get_static_emr_resources(),
        "Notes": f"numTrees={best_model.getOrDefault('numTrees')}, maxDepth={best_model.getOrDefault('maxDepth')}, minInstancesPerNode={best_model.getOrDefault('minInstancesPerNode')}"
    }

# Run for all emotions
emotions = ["joy", "sadness", "anger", "fear", "surprise", "disgust"]
rf_random_results = [run_rf_randomsearch_for_emotion(label) for label in emotions]

# Save as CSV
df_random_rf = pd.DataFrame(rf_random_results)
local_path = "/tmp/spark_rf_randomsearch_results.csv"
df_random_rf.to_csv(local_path, index=False)

# Upload to S3
s3 = boto3.client("s3")
bucket = "vera-final-project-bucket"
key = "gridsearch-results/spark_rf_randomsearch_results.csv"
s3.upload_file(local_path, bucket, key)

print("Random Search results uploaded to S3.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

JOY Random Search: AUC = 0.7634, Time = 187.05s
SADNESS Random Search: AUC = 0.7372, Time = 131.74s
ANGER Random Search: AUC = 0.9995, Time = 157.16s
FEAR Random Search: AUC = 0.7382, Time = 176.57s
SURPRISE Random Search: AUC = 0.7970, Time = 161.15s
DISGUST Random Search: AUC = 0.9860, Time = 124.67s
Random Search results uploaded to S3.

#### Archived MLP Parameter Grid (Too many grids may deplete the job)

In [12]:
# from pyspark.ml.classification import MultilayerPerceptronClassifier

# def run_mlp_for_emotion(emotion_label):
#     df = df_base.withColumn("label", when(col("emotion_clean").contains(emotion_label), 1).otherwise(0))
    
#     df = assembler.transform(df)
#     df = scaler.fit(df).transform(df)
    
#     input_dim = len(feature_cols)
#     mlp = MultilayerPerceptronClassifier(labelCol="label", featuresCol="features", seed=42)

#     paramGrid = ParamGridBuilder() \
#         .addGrid(mlp.maxIter, [100, 200]) \
#         .addGrid(mlp.layers, [
#             [input_dim, 50, 2],
#             [input_dim, 100, 50, 2],
#             [input_dim, 64, 32, 2]
#         ]) \
#         .build()

#     evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

#     cv = CrossValidator(
#         estimator=Pipeline(stages=[mlp]),
#         estimatorParamMaps=paramGrid,
#         evaluator=evaluator,
#         numFolds=3,
#         parallelism=2
#     )

#     import time
#     start = time.time()
#     cv_model = cv.fit(df)
#     duration = time.time() - start

#     best_model = cv_model.bestModel.stages[-1]
#     best_score = evaluator.evaluate(cv_model.transform(df))

#     print(f"{emotion_label.upper()} finished: AUC = {best_score:.4f}, time = {duration:.2f}s")
#     return {
#         "Platform": "Spark",
#         "Method": "Grid Search",
#         "Model": "MLP",
#         "Label": emotion_label,
#         "Runtime (s)": round(duration, 2),
#         "Best Score": round(best_score, 4),
#         "Resources Used": "EMR, m5.xlarge x 3",
#         "Notes": f"layers={best_model.getLayers()}, maxIter={best_model.getMaxIter()}"
#     }


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
# emotion_list = ["joy", "sadness", "anger", "fear", "surprise", "disgust"]
# mlp_results = []

# for emo in emotion_list:
#     print(f"Running MLP tuning for: {emo}")
#     mlp_results.append(run_mlp_for_emotion(emo))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [14]:
# import pandas as pd
# import boto3

# df_mlp = pd.DataFrame(mlp_results)
# local_path = "/tmp/spark_mlp_gridsearch_results.csv"
# df_mlp.to_csv(local_path, index=False)

# s3 = boto3.client('s3')
# bucket_name = 'vera-final-project-bucket'
# output_key = 'gridsearch-results/spark_mlp_gridsearch_results.csv'

# with open(local_path, 'rb') as f:
#     s3.put_object(Bucket=bucket_name, Key=output_key, Body=f)

# print(f"MLP results uploaded to s3://{bucket_name}/{output_key}")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [15]:
# from pyspark.ml.classification import MultilayerPerceptronClassifier
# from pyspark.ml import Pipeline
# from pyspark.ml.tuning import CrossValidator
# from pyspark.ml.evaluation import BinaryClassificationEvaluator
# import itertools
# import random
# import pandas as pd
# import boto3
# import time

# def run_mlp_randomsearch_for_emotion(emotion_label):
#     # Filter and label dataset
#     df = df_base.withColumn("label", when(col("emotion_clean").contains(emotion_label), 1).otherwise(0))
#     df = assembler.transform(df)
#     df = scaler.fit(df).transform(df)

#     # Define input and output size
#     input_dim = len(feature_cols)
#     output_dim = 2  # binary classification

#     # Define model
#     mlp = MultilayerPerceptronClassifier(labelCol="label", featuresCol="features", seed=42, maxIter=300)

#     # Define search space
#     param_space = {
#         mlp.layers: [
#             [input_dim, 64, output_dim],
#             [input_dim, 128, 64, output_dim],
#             [input_dim, 32, 16, output_dim],
#             [input_dim, 50, 50, output_dim]
#         ],
#         mlp.stepSize: [0.01, 0.05, 0.1],
#         mlp.blockSize: [64, 128, 256]
#     }

#     # Sample 10 combinations
#     all_combos = list(itertools.product(*param_space.values()))
#     sampled_combos = random.sample(all_combos, k=10)

#     param_maps = []
#     for combo in sampled_combos:
#         param_maps.append({
#             list(param_space.keys())[0]: combo[0],
#             list(param_space.keys())[1]: combo[1],
#             list(param_space.keys())[2]: combo[2]
#         })

#     evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

#     cv = CrossValidator(
#         estimator=Pipeline(stages=[mlp]),
#         estimatorParamMaps=param_maps,
#         evaluator=evaluator,
#         numFolds=3,
#         parallelism=2
#     )

#     start = time.time()
#     cv_model = cv.fit(df)
#     duration = time.time() - start

#     best_model = cv_model.bestModel.stages[-1]
#     best_score = evaluator.evaluate(cv_model.transform(df))

#     # Extract best params
#     best_layers = best_model.getLayers()
#     best_step = best_model.getStepSize()
#     best_block = best_model.getBlockSize()

#     print(f"{emotion_label.upper()} MLP Random Search: AUC = {best_score:.4f}, Time = {duration:.2f}s")

#     return {
#         "Platform": "Spark",
#         "Method": "Random Search",
#         "Model": "MLP",
#         "Label": emotion_label,
#         "Runtime (s)": round(duration, 2),
#         "Best Score": round(best_score, 4),
#         "Resources Used": "EMR, m5.xlarge x 3",
#         "Notes": f"layers={best_layers}, stepSize={best_step}, blockSize={best_block}"
#     }

# # Run for all emotions
# emotions = ["joy", "sadness", "anger", "fear", "surprise", "disgust"]
# mlp_random_results = [run_mlp_randomsearch_for_emotion(label) for label in emotions]

# # Save results
# df_mlp_random = pd.DataFrame(mlp_random_results)
# local_path = "/tmp/spark_mlp_randomsearch_results.csv"
# df_mlp_random.to_csv(local_path, index=False)

# # Upload to S3
# s3 = boto3.client("s3")
# bucket = "vera-final-project-bucket"
# key = "gridsearch-results/spark_mlp_randomsearch_results.csv"
# s3.upload_file(local_path, bucket, key)

# print("MLP Random Search results uploaded to S3.")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

#### MLP Parameter Grid

In [ ]:
# MLP - Grid Search (Fast)
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import pandas as pd
import boto3
import time

def run_mlp_gridsearch_for_emotion_fast(emotion_label):
    df = df_base.withColumn("label", when(col("emotion_clean").contains(emotion_label), 1).otherwise(0))
    df = assembler.transform(df)
    df = scaler.fit(df).transform(df)

    input_dim = len(feature_cols)
    output_dim = 2

    mlp = MultilayerPerceptronClassifier(labelCol="label", featuresCol="features", seed=42)

    paramGrid = ParamGridBuilder() \
        .addGrid(mlp.maxIter, [150]) \
        .addGrid(mlp.layers, [
            [input_dim, 100, output_dim],
            [input_dim, 100, 50, output_dim],
            [input_dim, 128, 64, 32, output_dim]
        ]) \
        .build()

    evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

    cv = CrossValidator(
        estimator=Pipeline(stages=[mlp]),
        estimatorParamMaps=paramGrid,
        evaluator=evaluator,
        numFolds=2,
        parallelism=2
    )

    start = time.time()
    cv_model = cv.fit(df)
    duration = time.time() - start

    best_model = cv_model.bestModel.stages[-1]
    best_score = evaluator.evaluate(cv_model.transform(df))

    return {
        "Platform": "Spark",
        "Method": "Grid Search (Fast)",
        "Model": "MLP",
        "Label": emotion_label,
        "Runtime (s)": round(duration, 2),
        "Best Score": round(best_score, 4),
        "Resources Used": get_static_emr_resources(),
        "Notes": f"layers={best_model.getLayers()}, maxIter={best_model.getMaxIter()}"
    }

emotions = ["joy", "sadness", "anger", "fear", "surprise", "disgust"]
mlp_grid_fast_results = [run_mlp_gridsearch_for_emotion_fast(label) for label in emotions]

df_grid = pd.DataFrame(mlp_grid_fast_results)
df_grid.to_csv("/tmp/spark_mlp_gridsearch_fast_results.csv", index=False)

boto3.client("s3").upload_file(
    "/tmp/spark_mlp_gridsearch_fast_results.csv",
    "vera-final-project-bucket",
    "gridsearch-results/spark_mlp_gridsearch_fast_results.csv"
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# MLP - Random Search (Fast)
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import itertools
import random
import pandas as pd
import boto3
import time

def run_mlp_randomsearch_for_emotion_fast(emotion_label):
    df = df_base.withColumn("label", when(col("emotion_clean").contains(emotion_label), 1).otherwise(0))
    df = assembler.transform(df)
    df = scaler.fit(df).transform(df)

    input_dim = len(feature_cols)
    output_dim = 2

    mlp = MultilayerPerceptronClassifier(labelCol="label", featuresCol="features", seed=42, maxIter=150)

    param_space = {
        mlp.layers: [
            [input_dim, 64, output_dim],
            [input_dim, 128, 64, output_dim],
            [input_dim, 32, 16, output_dim],
            [input_dim, 50, 50, output_dim]
        ],
        mlp.stepSize: [0.01, 0.05],
        mlp.blockSize: [64, 128]
    }

    all_combos = list(itertools.product(*param_space.values()))
    sampled_combos = random.sample(all_combos, k=6)

    param_maps = []
    for combo in sampled_combos:
        param_maps.append({
            list(param_space.keys())[0]: combo[0],
            list(param_space.keys())[1]: combo[1],
            list(param_space.keys())[2]: combo[2]
        })

    evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

    cv = CrossValidator(
        estimator=Pipeline(stages=[mlp]),
        estimatorParamMaps=param_maps,
        evaluator=evaluator,
        numFolds=2,
        parallelism=2
    )

    start = time.time()
    cv_model = cv.fit(df)
    duration = time.time() - start

    best_model = cv_model.bestModel.stages[-1]
    best_score = evaluator.evaluate(cv_model.transform(df))

    return {
        "Platform": "Spark",
        "Method": "Random Search (Fast)",
        "Model": "MLP",
        "Label": emotion_label,
        "Runtime (s)": round(duration, 2),
        "Best Score": round(best_score, 4),
        "Resources Used": get_static_emr_resources(),
        "Notes": f"layers={best_model.getLayers()}, stepSize={best_model.getStepSize()}, blockSize={best_model.getBlockSize()}"
    }

emotions = ["joy", "sadness", "anger", "fear", "surprise", "disgust"]
mlp_random_fast_results = [run_mlp_randomsearch_for_emotion_fast(label) for label in emotions]

df_random = pd.DataFrame(mlp_random_fast_results)
df_random.to_csv("/tmp/spark_mlp_randomsearch_fast_results.csv", index=False)

boto3.client("s3").upload_file(
    "/tmp/spark_mlp_randomsearch_fast_results.csv",
    "vera-final-project-bucket",
    "gridsearch-results/spark_mlp_randomsearch_fast_results.csv"
)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

#### Final Save

In [ ]:
import pandas as pd
import boto3
from io import BytesIO

# S3 setup
s3 = boto3.client('s3')
bucket = 'vera-final-project-bucket'
base_path = 'gridsearch-results/'

# Define all 6 file keys and labels
files = {
    'RF Grid Search': 'spark_rf_gridsearch_results.csv',
    'RF Random Search': 'spark_rf_randomsearch_results.csv',
    'MLP Grid Search': 'spark_mlp_gridsearch_results.csv',
    'MLP Grid Search (Fast)': 'spark_mlp_gridsearch_fast_results.csv',
    'MLP Random Search (Fast)': 'spark_mlp_randomsearch_fast_results.csv'
}

# Read each CSV from S3 and tag origin
dataframes = []
for label, key in files.items():
    obj = s3.get_object(Bucket=bucket, Key=base_path + key)
    df = pd.read_csv(BytesIO(obj['Body'].read()))
    df['Source'] = label
    dataframes.append(df)

# Concatenate all results
df_combined = pd.concat(dataframes, ignore_index=True)

# Save locally
local_path = "/tmp/spark_method_comparison_results_EMR_m5xlargeX7.csv"
df_combined.to_csv(local_path, index=False)

# Upload back to S3
output_key = base_path + "spark_method_comparison_results_EMR_m5xlargeX7.csv"
s3.upload_file(local_path, bucket, output_key)

print(f"Combined results uploaded to s3://{bucket}/{output_key}")


In [ ]:
import boto3
import pandas as pd
from io import BytesIO

s3 = boto3.client("s3")
bucket = "vera-final-project-bucket"
base_path = "gridsearch-results/"

files_to_merge = [
    "spark_method_comparison_results_EMR_m5xlargeX3.csv",
    "spark_method_comparison_results_EMR_m5xlargeX5.csv",
    "spark_method_comparison_results_EMR_r5xlargeX3.csv",
    "spark_method_comparison_results_EMR_m5xlargeX7.csv"
]

dataframes = []
for filename in files_to_merge:
    obj = s3.get_object(Bucket=bucket, Key=base_path + filename)
    df = pd.read_csv(BytesIO(obj['Body'].read()))
    df['Source File'] = filename  
    dataframes.append(df)

df_combined = pd.concat(dataframes, ignore_index=True)

local_path = "/tmp/spark_method_comparison_results.csv"
df_combined.to_csv(local_path, index=False)

output_key = base_path + "spark_method_comparison_results.csv"
s3.upload_file(local_path, bucket, output_key)

print(f"Combined results uploaded to s3://{bucket}/{output_key}")
